## 02 — Comparing the LOD Levels

The four output files exist. Now we load them and answer:

1. How do the files differ in size, feature count, and coordinate count?
2. What does each level look like on a real map?
3. At what zoom does each level look correct — and at what zoom does it start to look wrong?

This is the quality check before we build the switching logic.

## Load All Four Levels

In [ ]:
import json
from pathlib import Path

lod_dir = Path("../../data/lod")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

lod_data = {}
for name, filename in lod_files.items():
    path = lod_dir / filename
    with open(path) as f:
        lod_data[name] = json.load(f)
    print(f"Loaded {name}: {len(lod_data[name]['features']):,} features")

## Summary Table — Size and Coordinate Count

In [ ]:
print(f"{'Level':<12} {'Zoom':>6} {'Features':>10} {'Total pts':>12} {'File (MB)':>11}")
print("-" * 55)

zoom_ranges = {"coarse": "1-3", "medium": "4-6", "fine": "7-10", "extra_fine": "11+"}

for name, filename in lod_files.items():
    path = lod_dir / filename
    fc   = lod_data[name]
    n_features = len(fc["features"])
    total_pts  = sum(len(f["geometry"]["coordinates"]) for f in fc["features"])
    size_mb    = path.stat().st_size / 1_000_000
    zoom       = zoom_ranges[name]
    print(f"{name:<12} {zoom:>6} {n_features:>10,} {total_pts:>12,} {size_mb:>10.2f}")

# Also show the original for comparison
original_path = Path("../../data/ne_10m_railroads.geojson")
with open(original_path) as f:
    original = json.load(f)
orig_pts  = sum(len(f["geometry"]["coordinates"]) for f in original["features"])
orig_size = original_path.stat().st_size / 1_000_000
print("-" * 55)
print(f"{'original':<12} {'all':>6} {len(original['features']):>10,} {orig_pts:>12,} {orig_size:>10.2f}")

## Visual Comparison — One Level at a Time

Display each LOD level on a map. Pan and zoom to see where it starts to look correct and where it breaks down.

In [ ]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

# Change this to switch between levels: "coarse", "medium", "fine", "extra_fine"
level = "coarse"

level_zoom = {"coarse": 2, "medium": 4, "fine": 7, "extra_fine": 10}

m = Map(center=[20, 0], zoom=level_zoom[level])
layer = GeoJSON(
    data=lod_data[level],
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

print(f"Showing: {level}  |  {len(lod_data[level]['features']):,} features")
m

Try each level at its intended zoom range. Then deliberately zoom in too far on the coarse level — you should see the simplification artifacts clearly: straight lines where there should be curves.

## Side-by-Side Comparison — One Region

To see the difference sharply, let's crop to a small geographic area and display all four levels.

We filter to features whose bounding box falls within Europe (roughly).

In [ ]:
import matplotlib.pyplot as plt

# Europe bounding box [lon_min, lat_min, lon_max, lat_max]
europe = (-10, 35, 40, 70)

def coords_in_bbox(coords, bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    return any(
        lon_min <= c[0] <= lon_max and lat_min <= c[1] <= lat_max
        for c in coords
    )

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
titles = ["Coarse (ε=1.0)", "Medium (ε=0.1)", "Fine (ε=0.01)", "Extra Fine (ε=0.001)"]

for ax, (name, _), title in zip(axes, lod_files.items(), titles):
    europe_features = [
        f for f in lod_data[name]["features"]
        if coords_in_bbox(f["geometry"]["coordinates"], europe)
    ]
    for f in europe_features:
        xs = [c[0] for c in f["geometry"]["coordinates"]]
        ys = [c[1] for c in f["geometry"]["coordinates"]]
        ax.plot(xs, ys, '-', color='#cc3300', linewidth=0.6, alpha=0.7)
    ax.set_xlim(europe[0], europe[2])
    ax.set_ylim(europe[1], europe[3])
    ax.set_title(f"{title}\n{len(europe_features):,} features")
    ax.set_aspect('equal')

plt.suptitle('European railroads — four LOD levels compared', y=1.02)
plt.tight_layout()
plt.show()

The coarse level will show far fewer features (scalerank filter) and angular lines. The extra fine level should look nearly identical to the original.

## Exercise A

Pick one specific railroad feature that appears in all four LOD files — use the `rwdb_rr_id` property to find the same feature across files.

Plot that single feature at all four simplification levels on one chart. Label each with its point count.

In [ ]:
# Find a feature by rwdb_rr_id that exists in all four LOD levels
# Plot it at all four simplification levels
# Your code here


def build_id_index(fc):
    return {
        f["properties"]["rwdb_rr_id"]: f
        for f in fc["features"]
        if "rwdb_rr_id" in f["properties"]
    }

indexes = {name: build_id_index(lod_data[name]) for name in lod_data}

# Find IDs that appear in ALL four levels
common_ids = set(indexes["extra_fine"].keys())
for name in indexes:
    common_ids &= set(indexes[name].keys())

print(f"Features present in all four LOD levels: {len(common_ids):,}")

# Pick the first common ID (or change this to explore others)
target_id = sorted(common_ids)[0]
print(f"Plotting rwdb_rr_id = {target_id}")

# Plot that single feature at all four LOD levels
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
level_labels = {
    "coarse":     "Coarse (ε=1.0)",
    "medium":     "Medium (ε=0.1)",
    "fine":       "Fine (ε=0.01)",
    "extra_fine": "Extra Fine (ε=0.001)",
}
colors = ["#cc3300", "#e07b00", "#007acc", "#228b22"]

for ax, (name, label), color in zip(axes, level_labels.items(), colors):
    feature = indexes[name][target_id]
    coords  = feature["geometry"]["coordinates"]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]

    ax.plot(xs, ys, '-o', color=color, linewidth=1.5,
            markersize=3, alpha=0.85)
    ax.set_title(f"{label}\n{len(coords)} points")
    ax.set_aspect("equal")
    ax.tick_params(labelsize=7)

plt.suptitle(f"Single railroad feature across LOD levels\n(rwdb_rr_id = {target_id})", y=1.03)
plt.tight_layout()
plt.show()

## Exercise B

Calculate the **compression ratio** for each LOD level — the ratio of original coordinate count to simplified coordinate count for the features they share.

Then answer: which level gives the best size reduction per unit of visual quality loss?

In [ ]:
# Calculate compression ratio per LOD level
# Your code here

# Build original index by rwdb_rr_id
orig_index = {
    f["properties"]["rwdb_rr_id"]: f
    for f in original["features"]
    if "rwdb_rr_id" in f["properties"]
}

print(f"{'Level':<12} {'Shared':>8} {'Orig pts':>10} {'Simp pts':>10} {'Ratio':>8} {'% kept':>8}")
print("-" * 60)

ratios = {}
for name in ["coarse", "medium", "fine", "extra_fine"]:
    idx = indexes[name]
    shared_ids = set(idx.keys()) & set(orig_index.keys())

    orig_total = sum(len(orig_index[i]["geometry"]["coordinates"]) for i in shared_ids)
    simp_total = sum(len(idx[i]["geometry"]["coordinates"])         for i in shared_ids)

    ratio   = orig_total / simp_total if simp_total else float("inf")
    pct     = (simp_total / orig_total * 100) if orig_total else 0
    ratios[name] = {"ratio": ratio, "pct_kept": pct, "shared": len(shared_ids)}

    print(f"{name:<12} {len(shared_ids):>8,} {orig_total:>10,} {simp_total:>10,} {ratio:>8.2f}x {pct:>7.1f}%")


fig, ax = plt.subplots(figsize=(7, 4))
names  = list(ratios.keys())
values = [ratios[n]["ratio"] for n in names]
bars   = ax.bar(names, values, color=["#cc3300","#e07b00","#007acc","#228b22"], alpha=0.85)
ax.bar_label(bars, fmt="%.1fx", padding=3, fontsize=10)
ax.set_ylabel("Compression ratio (original pts / simplified pts)")
ax.set_title("LOD compression ratio by level\n(higher = more aggressive simplification)")
ax.axhline(1, color="black", linewidth=0.8, linestyle="--", label="no compression")
ax.legend()
plt.tight_layout()
plt.show()

## Check Your Understanding

The coarse level has far fewer features than the other levels because of the `scalerank <= 4` filter.

If a user zooms into a region that has no coarse-level features (e.g. a small country whose railroads are all scalerank 5+), what will they see? And what does this tell us about a limitation of the simple scalerank filter approach?

Write a 2–3 sentence answer.

The coarse LOD level will show a completely blank map for that aread, no railroad lines at all even though there exists one there. This reveals a limitation of scalarrank

---

## Next

In [Module 03 — Bounding Box Culling](../03-Bounding_Box_Culling/README.md), we add viewport filtering — so we only render the features the user can actually see.